In [1]:
import pandas as pd
import numpy as np
import re
import os
from datetime import datetime
import src.datetime_utils as dateTime

from src.EqCat import EqCat

In [2]:
# функция для загрузки каталога

def load_isc_catalog(file_path):

    # Read all lines from file
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    # Find data section (skip header and stop at 'STOP')
    start_idx = 21  # Skip first 21 header lines
    end_idx = len(lines)
    
    # Find where data ends (at 'STOP')
    for i in range(start_idx, len(lines)):
        if lines[i].strip().upper() == 'STOP':
            end_idx = i
            break
    
    # Process data lines
    events = []
    magnitudes = []
    
    for i in range(start_idx, end_idx):
        line = lines[i].strip()
        if not line or line.startswith('#'):
            continue
            
        # Remove trailing comma if present
        if line.endswith(','):
            line = line[:-1]
            
        # Split by comma
        parts = [part.strip() for part in line.split(',')]
        
        # Skip malformed lines
        if len(parts) < 9 or not parts[0].replace('.', '').isdigit():
            continue
        
        try:
            # Extract basic event information (first 9 fields)
            event_id = int(parts[0])
            event_type = parts[1].strip()
            author = parts[2].strip()
            date = parts[3].strip()
            time = parts[4].strip()
            
            # Parse coordinates and depth
            lat = float(parts[5]) if parts[5] else np.nan
            lon = float(parts[6]) if parts[6] else np.nan
            depth = float(parts[7]) if parts[7] else np.nan
            depfix = parts[8].strip()
            
            # Add to events list (ONLY basic event data, no magnitudes)
            events.append({
                'EVENTID': event_id,
                'TYPE': event_type,
                'AUTHOR': author,
                'DATE': date,
                'TIME': time,
                'LAT': lat,
                'LON': lon,
                'DEPTH': depth,
                'DEPFIX': depfix
            })
            
            # Process ALL magnitudes starting from position 9 (groups of 3)
            j = 9
            while j + 2 < len(parts):
                mag_author = parts[j].strip()
                mag_type = parts[j+1].strip()
                mag_value = parts[j+2].strip()
                
                # Only add valid magnitude values
                if mag_value and re.match(r'-?\d+(\.\d+)?', mag_value):
                    magnitudes.append({
                        'EVENTID': event_id,
                        'AUTHOR': mag_author,
                        'MAG_TYPE': mag_type,
                        'MAG_VALUE': float(mag_value)
                    })
                
                j += 3
                
        except (ValueError, IndexError) as e:
            print(f"Warning: Error parsing line {i+1}: {e}")
            print(f"Line content: {line[:100]}...")
    
    # Create DataFrames
    main_df = pd.DataFrame(events)
    magnitudes_df = pd.DataFrame(magnitudes) if magnitudes else pd.DataFrame(columns=['EVENTID', 'AUTHOR', 'MAG_TYPE', 'MAG_VALUE'])
    
    print(f"Successfully loaded {len(main_df)} earthquake events")
    if not magnitudes_df.empty:
        print(f"Loaded {len(magnitudes_df)} magnitude measurements from {magnitudes_df['AUTHOR'].nunique()} agencies")
    
    return main_df, magnitudes_df

In [3]:
file_path = 'data/Tohoku_eqs.txt'

main_df, magnitudes_df = load_isc_catalog(file_path)

neic_magnitudes = magnitudes_df[magnitudes_df['AUTHOR'] == 'NEIC'].copy()

neic_magnitudes = neic_magnitudes.groupby('EVENTID').first().reset_index()

merged_df = pd.merge(main_df, neic_magnitudes, on='EVENTID', how='inner')

merged_df['time'] = pd.to_datetime(
    merged_df['DATE'] + ' ' + merged_df['TIME'].str.replace(',', ' '),
    errors='coerce'
)

print(merged_df)

def datetime_to_decimal_year(dt_series):
    def to_dec_year(t):
        if pd.isna(t):
            return np.nan
        return dateTime.dateTime2decYr([t.year, t.month, t.day, t.hour, t.minute, t.second])
    return np.array([to_dec_year(t) for t in dt_series])

eqCat = EqCat( )
n_events = len(merged_df)
eqCat.data = {
    'Time': datetime_to_decimal_year(merged_df['time']),
    'N': merged_df['EVENTID'].values,
    'Lat': merged_df['LAT'].values,
    'Lon': merged_df['LON'].values,
    'Depth': merged_df['DEPTH'].values,
    'Mag': merged_df['MAG_VALUE'].values
}

output_file = 'data/Tohoku_eqs.mat'
eqCat.saveMatBin(output_file)

Successfully loaded 54632 earthquake events
Loaded 122169 magnitude measurements from 9 agencies
        EVENTID TYPE AUTHOR_x        DATE         TIME      LAT       LON  \
0      16461282   de      ISC  2011-03-11  05:46:23.20  38.2963  142.4980   
1     602653817   se      ISC  2011-03-11  05:54:31.46  37.5085  141.3588   
2     602653811   fe      ISC  2011-03-11  05:55:45.18  37.2979  143.4095   
3     602653796   se      ISC  2011-03-11  05:58:02.32  37.6702  142.0447   
4      16403886   se      ISC  2011-03-11  05:59:35.25  37.0449  141.7449   
...         ...  ...      ...         ...          ...      ...       ...   
3430   16503598   se      ISC  2011-05-10  18:24:17.43  38.2697  141.9514   
3431   16560962   se      ISC  2011-05-10  19:22:25.15  35.7809  140.9689   
3432  600611467   se      ISC  2011-05-10  19:58:10.22  38.8860  141.9967   
3433   16503601   fe      ISC  2011-05-10  20:24:14.49  35.6571  141.1377   
3434   16503610   se      ISC  2011-05-11  01:20:07.32  